# JSON Basics — 01: Reading JSON Files

JSON is the most common format for REST APIs, logs, and NoSQL exports.
Spark reads JSON as newline-delimited (one JSON object per line) by default.

Topics: spark.read.json, inferSchema vs explicit, multiLine, corrupt records,
reading arrays of objects, options reference.


In [1]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob as glib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/json_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('json-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 05:59:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2


## Step 1 — Create Sample JSON Files



In [2]:

import pathlib, json as pyjson, random, datetime
random.seed(42)

ORDERS = [{"order_id":i, "customer_id":random.randint(1,1000),
           "product":f"Product_{random.randint(1,50)}",
           "category":random.choice(["Electronics","Books","Clothing","Food"]),
           "country":random.choice(["US","UK","DE","FR","JP"]),
           "quantity":random.randint(1,10),
           "price":round(random.uniform(5,1000),2),
           "status":random.choice(["pending","confirmed","shipped","delivered"]),
           "order_date":str(datetime.date(2024,1,1)+datetime.timedelta(days=random.randint(0,180)))}
          for i in range(1, 50_001)]

# Newline-delimited JSON (NDJSON) — one object per line
ndjson_path = f"{DATA_DIR}/orders.ndjson"
with open(ndjson_path,"w") as f:
    for row in ORDERS:
        f.write(pyjson.dumps(row)+"\n")

# Pretty-printed JSON array (multiLine)
pretty_path = f"{DATA_DIR}/orders_pretty.json"
with open(pretty_path,"w") as f:
    pyjson.dump(ORDERS[:100], f, indent=2)

# JSON with corrupt rows
corrupt_path = f"{DATA_DIR}/orders_corrupt.json"
with open(corrupt_path,"w") as f:
    for row in ORDERS[:500]:
        f.write(pyjson.dumps(row)+"\n")
    f.write("{broken json here without closing brace\n")
    f.write("not json at all\n")
    for row in ORDERS[500:1000]:
        f.write(pyjson.dumps(row)+"\n")

size_kb = pathlib.Path(ndjson_path).stat().st_size//1024
print(f"Created: orders.ndjson ({size_kb} KB, {len(ORDERS):,} rows)")
print(f"Created: orders_pretty.json (100 rows, pretty-printed)")
print(f"Created: orders_corrupt.json (with 2 bad rows)")


Created: orders.ndjson (9100 KB, 50,000 rows)
Created: orders_pretty.json (100 rows, pretty-printed)
Created: orders_corrupt.json (with 2 bad rows)


## Step 2 — Basic Read: NDJSON



In [3]:

# Default: one JSON object per line
df = spark.read.json(f"{DATA_DIR}/orders.ndjson")
print(f"Rows: {df.count():,}")
df.printSchema()
df.show(5, truncate=False)
print("Spark infers schema from first 1000 rows by default (samplingRatio=1.0)")


Rows: 50,000
root
 |-- category: string (nullable = true)
 |-- country: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_id: long (nullable = true)
 |-- price: double (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- status: string (nullable = true)

+-----------+-------+-----------+----------+--------+------+----------+--------+---------+
|category   |country|customer_id|order_date|order_id|price |product   |quantity|status   |
+-----------+-------+-----------+----------+--------+------+----------+--------+---------+
|Electronics|DE     |655        |2024-06-22|1       |227.09|Product_8 |4       |pending  |
|Electronics|JP     |759        |2024-02-25|2       |36.62 |Product_35|7       |pending  |
|Electronics|JP     |239        |2024-02-26|3       |717.44|Product_33|4       |delivered|
|Clothing   |US     |460        |2024-03-12|4       |699.65|Product_38|3       |shi

## Step 3 — multiLine: JSON Arrays and Pretty-Printed Files



In [4]:

# multiLine=True: entire file is one JSON document (or pretty-printed)
df_multi = spark.read.option("multiLine", True).json(f"{DATA_DIR}/orders_pretty.json")
print(f"multiLine=True: {df_multi.count():,} rows")
df_multi.show(3)
print()
print("When to use multiLine=True:")
print("  - JSON array: [{...},{...}]")
print("  - Pretty-printed JSON (spans multiple lines)")
print("  - Single large JSON object")
print()
print("When NOT to use (default, multiLine=False):")
print("  - NDJSON/JSONL: one JSON object per line (most common!)")
print("  - Large files — multiLine disables parallel reads")


multiLine=True: 100 rows
+-----------+-------+-----------+----------+--------+------+----------+--------+---------+
|   category|country|customer_id|order_date|order_id| price|   product|quantity|   status|
+-----------+-------+-----------+----------+--------+------+----------+--------+---------+
|Electronics|     DE|        655|2024-06-22|       1|227.09| Product_8|       4|  pending|
|Electronics|     JP|        759|2024-02-25|       2| 36.62|Product_35|       7|  pending|
|Electronics|     JP|        239|2024-02-26|       3|717.44|Product_33|       4|delivered|
+-----------+-------+-----------+----------+--------+------+----------+--------+---------+
only showing top 3 rows

When to use multiLine=True:
  - JSON array: [{...},{...}]
  - Pretty-printed JSON (spans multiple lines)
  - Single large JSON object

When NOT to use (default, multiLine=False):
  - NDJSON/JSONL: one JSON object per line (most common!)
  - Large files — multiLine disables parallel reads


## Step 4 — Corrupt Record Handling



In [5]:

# PERMISSIVE mode (default) captures corrupt rows
schema_with_corrupt = spark.read.json(f"{DATA_DIR}/orders.ndjson").schema
from pyspark.sql.types import StructField, StringType
schema_with_corrupt = schema_with_corrupt.add(StructField("_corrupt_record", StringType()))

df_corrupt = spark.read \
    .option("mode", "PERMISSIVE") \
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .schema(schema_with_corrupt) \
    .json(f"{DATA_DIR}/orders_corrupt.json")

good = df_corrupt.filter(F.col("_corrupt_record").isNull()).count()
bad  = df_corrupt.filter(F.col("_corrupt_record").isNotNull())
print(f"Good rows: {good:,}")
print(f"Bad rows : {bad.count()}")
print()
print("Corrupt rows:")
bad.select("_corrupt_record").show(truncate=False)


AnalysisException: [UNSUPPORTED_FEATURE.QUERY_ONLY_CORRUPT_RECORD_COLUMN] The feature is not supported: Queries from raw JSON/CSV/XML files are disallowed when the
referenced columns only include the internal corrupt record column
(named `_corrupt_record` by default). For example:
`spark.read.schema(schema).json(file).filter($"_corrupt_record".isNotNull).count()`
and `spark.read.schema(schema).json(file).select("_corrupt_record").show()`.
Instead, you can cache or save the parsed results and then send the same query.
For example, `val df = spark.read.schema(schema).json(file).cache()` and then
`df.filter($"_corrupt_record".isNotNull).count()`. SQLSTATE: 0A000

## Step 5 — Read Options Reference



In [ ]:

print("""
Common spark.read.json() options:

  multiLine          = False  # True for pretty-printed / JSON arrays
  primitivesAsString = False  # True to read all primitives as strings
  prefersDecimal     = False  # True to infer decimals instead of doubles
  allowComments      = False  # True to allow # and // comments in JSON
  allowUnquotedFieldNames = False  # relaxed JSON
  allowSingleQuotes  = False  # allow 'field': 'value'
  allowNumericLeadingZeros = False
  allowBackslashEscapingAnyCharacter = False
  mode               = PERMISSIVE   # PERMISSIVE | DROPMALFORMED | FAILFAST
  columnNameOfCorruptRecord = _corrupt_record
  dateFormat         = yyyy-MM-dd
  timestampFormat    = yyyy-MM-dd'T'HH:mm:ss[.SSS][XXX]
  samplingRatio      = 1.0  # fraction of rows used for schema inference
  dropFieldIfAllNull = False  # drop column if all values are null
  encoding           = UTF-8
  locale             = en-US
""")


## Step 6 — Read Multiple JSON Files



In [ ]:

# Create multiple JSON files
multi_dir = f"{DATA_DIR}/multi_json"
pathlib.Path(multi_dir).mkdir(exist_ok=True)
for month in range(1, 5):
    rows = [{"order_id":month*10000+i,"month":month,
             "revenue":round(random.uniform(10,500),2)}
            for i in range(1000)]
    with open(f"{multi_dir}/orders_2024_{month:02d}.json","w") as f:
        for r in rows: f.write(pyjson.dumps(r)+"\n")

# Read all files in directory
df_all = spark.read.json(f"{multi_dir}/")
print(f"All files in directory: {df_all.count():,} rows")

# Glob pattern
df_q1 = spark.read.json(f"{multi_dir}/orders_2024_0[123].json")
print(f"Q1 only (glob): {df_q1.count():,} rows")

# Add source filename
df_with_source = spark.read.json(f"{multi_dir}/") \
    .withColumn("source_file", F.input_file_name())
df_with_source.select("order_id","month","source_file").show(4, truncate=False)


## Summary



In [ ]:

print("""
spark.read.json() quick reference:

  # NDJSON (one object per line)
  spark.read.json("/path/orders.ndjson")

  # JSON array or pretty-printed
  spark.read.option("multiLine", True).json("/path/file.json")

  # With corrupt record handling
  spark.read
       .option("mode","PERMISSIVE")
       .option("columnNameOfCorruptRecord","_corrupt_record")
       .schema(my_schema_with_corrupt_col)
       .json("/path/")

  # With explicit schema (no inferSchema cost)
  spark.read.schema(my_schema).json("/path/")

  # Multiple files
  spark.read.json("/path/dir/", "/path/other/")
  spark.read.json("/path/*.json")  # glob
""")
